## Kaggle Homework- Pranika Chandra-@PranikaC

## Exploratory Data Analysis

Importing required packages prior to performing Exploratory Data Analysis

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sweetviz as sv
from sklearn.model_selection import train_test_split # for boosting/bagging modeling
from xgboost import XGBClassifier # for boosting/bagging modeling
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # for boosting/bagging modeling
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

Import + View the training dataset 

In [48]:
df_train = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\train.csv')
df_test = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\test.csv')

In [49]:
X = df_train.drop('Irrigation_Need', axis=1)
y = df_train['Irrigation_Need'].copy()

In [50]:
df_train.head(5)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Performing Exploratory Data Analysis using SweetViz

In [19]:
report = sv.analyze(df_train)
report.show_html()

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)


Report SWEETVIZ_REPORT.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


What did the EDA tell us?

What insights did you gain? Is there a feature you think is especially important? Any potential issues?

Based on the EDA, we can see that there are 630,000 null values and that there are no strong direct relationships numerically nor categorically. Around 60% of the data represents low irrigation_need, whereas less than 10% of the data shows high irrigation need. The features look evenly distributed and therefore shows no clear, contributing features.

## Boosting Model

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

best_model = GradientBoostingClassifier(
    n_estimators=10,
    learning_rate=0.1,
    max_depth=2,
    random_state=42
)

print("Training final model...")
best_model.fit(X_train, y_train)
print("Training complete.")

y_pred = best_model.predict(X_valid)
print(classification_report(y_valid, y_pred))

test_pred = best_model.predict(X_test_final)

submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': test_pred
})

submission.to_csv('GradientBoosting_submission.csv', index=False)
print("Saved.")

Training final model...
Training complete.
              precision    recall  f1-score   support

        High       0.80      0.30      0.43      4202
         Low       0.76      0.94      0.84     73983
      Medium       0.78      0.53      0.63     47815

    accuracy                           0.76    126000
   macro avg       0.78      0.59      0.63    126000
weighted avg       0.77      0.76      0.75    126000

Saved.


# Bagging Model

In [67]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10, 
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)
print("Training complete.")

# =========================
# VALIDATION
# =========================
y_pred_rf = rf_model.predict(X_valid)

print("Validation Accuracy:", accuracy_score(y_valid, y_pred_rf))
print(classification_report(y_valid, y_pred_rf))

# =========================
# TEST PREDICTIONS
# =========================
print("Predicting test set...")
test_pred_rf = rf_model.predict(X_test_final)

# =========================
# SAVE SUBMISSION
# =========================
submission_rf = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': test_pred_rf
})

submission_rf.to_csv('submission_rf.csv', index=False)
print("submission_rf.csv created successfully")
print(submission_rf.head())

Training Random Forest...
Training complete.
Validation Accuracy: 0.972468253968254
              precision    recall  f1-score   support

        High       0.99      0.50      0.67      4202
         Low       0.99      1.00      0.99     73983
      Medium       0.95      0.98      0.96     47815

    accuracy                           0.97    126000
   macro avg       0.97      0.83      0.87    126000
weighted avg       0.97      0.97      0.97    126000

Predicting test set...
submission_rf.csv created successfully
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low


# Comparing the two models:

In [70]:
y_pred_gb = best_model.predict(X_valid)

rf_accuracy = accuracy_score(y_valid, y_pred_rf)
gb_accuracy = accuracy_score(y_valid, y_pred_gb)

rf_f1 = f1_score(y_valid, y_pred_rf, average='macro')
gb_f1 = f1_score(y_valid, y_pred_gb, average='macro')

print("Bagging vs Boosting Model Comparison")
print("-----------------------------------")
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Gradient Boosting Accuracy: {gb_accuracy:.4f}")
print(f"Random Forest Macro F1: {rf_f1:.4f}")
print(f"Gradient Boosting Macro F1: {gb_f1:.4f}")

Bagging vs Boosting Model Comparison
-----------------------------------
Random Forest Accuracy: 0.9725
Gradient Boosting Accuracy: 0.7630
Random Forest Macro F1: 0.8743
Gradient Boosting Macro F1: 0.6346
